# LoRA Target Module Selection

## Historical Context: Where to Apply LoRA?

### The Evolution of Target Module Selection

**Early LoRA (2021):**
- Original paper: Applied only to attention weights (Q, V)
- Rationale: Attention is where most adaptation happens
- Result: Good performance, minimal parameters

**Empirical Discoveries (2022-2023):**
- Community experiments: Different tasks benefit from different targets
- Findings:
  - Classification: Attention-only is sufficient
  - Generation: All attention + FFN improves quality
  - Domain adaptation: Broader coverage helps

### The Core Question

**Where should we apply LoRA?**
```
Transformer Layer:
├── Attention Block
│   ├── q_proj (Query projection)      ← Apply LoRA?
│   ├── k_proj (Key projection)        ← Apply LoRA?
│   ├── v_proj (Value projection)      ← Apply LoRA?
│   └── o_proj (Output projection)     ← Apply LoRA?
└── Feed-Forward Network (FFN)
    ├── gate_proj (or fc1/w1)          ← Apply LoRA?
    ├── up_proj (or fc2/w2)            ← Apply LoRA?
    └── down_proj (or fc3/w3)          ← Apply LoRA?
```

**Trade-offs:**
- More modules = More capacity, Better quality
- More modules = More parameters, More memory
- More modules = Longer training time

## Problem Statement

Determine optimal target modules for:
1. Classification tasks
2. Generation tasks
3. Domain adaptation
4. Different model architectures (BERT, GPT, Llama)
5. Memory-constrained scenarios

In [ ]:
# Setup
import torch
import torch.nn as nn
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer
)
from peft import LoraConfig, get_peft_model, TaskType
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple

print("PyTorch version:", torch.__version__)

## Step 1: Understanding Model Architecture

### Llama 2 Architecture

Each transformer layer contains:

**Attention Block (Multi-Head Attention):**
```python
# Shapes for Llama 2 7B (hidden_size=4096, num_heads=32)
q_proj: Linear(4096, 4096)  # Query projection
k_proj: Linear(4096, 4096)  # Key projection  
v_proj: Linear(4096, 4096)  # Value projection
o_proj: Linear(4096, 4096)  # Output projection

Total attention params: 4 × 4096 × 4096 = 67M per layer
```

**Feed-Forward Network (SwiGLU):**
```python
# Shapes for Llama 2 7B (intermediate_size=11008)
gate_proj: Linear(4096, 11008)  # Gate for SwiGLU
up_proj: Linear(4096, 11008)    # Up projection
down_proj: Linear(11008, 4096)  # Down projection

Total FFN params: 2 × 4096 × 11008 + 11008 × 4096 = 135M per layer
```

**Per-Layer Breakdown:**
- Attention: 67M parameters (33%)
- FFN: 135M parameters (67%)
- 32 layers total → 6.7B parameters

**Key Insight:** FFN has 2x more parameters than attention!

In [ ]:
# Inspect Llama model structure
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Smaller model for demo
print(f"Loading {model_id}...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu"  # Keep on CPU for inspection
)

print("\n=== Model Layer Structure ===")
print("\nFirst layer components:")

first_layer = model.model.layers[0]
print("\nAttention block:")
for name, module in first_layer.self_attn.named_children():
    if isinstance(module, nn.Linear):
        print(f"  {name}: {module.weight.shape}")

print("\nFeed-forward network:")
for name, module in first_layer.mlp.named_children():
    if isinstance(module, nn.Linear):
        print(f"  {name}: {module.weight.shape}")

# Count parameters
def count_params(module):
    return sum(p.numel() for p in module.parameters())

attn_params = count_params(first_layer.self_attn)
ffn_params = count_params(first_layer.mlp)
total_params = attn_params + ffn_params

print(f"\n=== Parameter Breakdown (First Layer) ===")
print(f"Attention: {attn_params:,} ({100*attn_params/total_params:.1f}%)")
print(f"FFN: {ffn_params:,} ({100*ffn_params/total_params:.1f}%)")
print(f"Total: {total_params:,}")

## Step 2: Target Module Configurations

### Common LoRA Configurations

**1. Minimal (Q, V only):**
```python
target_modules = ["q_proj", "v_proj"]
```
- Original LoRA paper configuration
- 50% of attention parameters
- Best for: Classification, Q&A
- Memory: Minimal

**2. All Attention (Q, K, V, O):**
```python
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
```
- Full attention coverage
- Standard choice for most tasks
- Best for: General purpose
- Memory: Moderate

**3. Attention + FFN:**
```python
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # Attention
    "gate_proj", "up_proj", "down_proj"      # FFN
]
```
- Maximum coverage
- Best for: Generation, domain adaptation
- Memory: 3x more than attention-only

**4. FFN only:**
```python
target_modules = ["gate_proj", "up_proj", "down_proj"]
```
- Experimental configuration
- Best for: Some vision tasks
- Memory: 2x more than attention-only

In [ ]:
# Calculate LoRA parameters for different configurations
def calculate_lora_params(hidden_size: int, intermediate_size: int, 
                         rank: int, num_layers: int, 
                         target_modules: List[str]) -> Dict:
    """
    Calculate LoRA parameter count for different target module configs.
    
    Args:
        hidden_size: Model hidden dimension
        intermediate_size: FFN intermediate dimension
        rank: LoRA rank
        num_layers: Number of transformer layers
        target_modules: List of target module names
    
    Returns:
        Dictionary with parameter counts and memory estimates
    """
    params = 0
    
    # Attention modules (hidden_size x hidden_size)
    attention_modules = {"q_proj", "k_proj", "v_proj", "o_proj"}
    for module in target_modules:
        if module in attention_modules:
            # LoRA: A (hidden_size, rank) + B (rank, hidden_size)
            params += 2 * hidden_size * rank
    
    # FFN modules
    if "gate_proj" in target_modules:
        params += 2 * hidden_size * rank  # (hidden_size, rank) + (rank, intermediate_size)
        params += 2 * intermediate_size * rank  # But simplified: approximate as up+gate
    if "up_proj" in target_modules:
        params += 2 * hidden_size * rank
        params += 2 * intermediate_size * rank
    if "down_proj" in target_modules:
        params += 2 * intermediate_size * rank
        params += 2 * hidden_size * rank
    
    # Multiply by number of layers
    total_params = params * num_layers
    
    # Memory (FP16 = 2 bytes)
    # Training: params + gradients + optimizer (2x for Adam)
    memory_mb = (total_params * 2 * 4) / (1024 * 1024)
    
    return {
        "params": total_params,
        "params_millions": total_params / 1e6,
        "memory_mb": memory_mb
    }

# Test configurations for Llama 2 7B
hidden_size = 4096
intermediate_size = 11008
rank = 16
num_layers = 32

configs = {
    "Q,V only": ["q_proj", "v_proj"],
    "All Attention": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "Attention + FFN": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "FFN only": ["gate_proj", "up_proj", "down_proj"]
}

print("=== LoRA Parameters by Configuration (Llama 2 7B, r=16) ===\n")
print(f"{'Configuration':<20} {'Parameters':<15} {'Memory (MB)':<12}")
print("-" * 50)

results = {}
for name, modules in configs.items():
    result = calculate_lora_params(hidden_size, intermediate_size, rank, num_layers, modules)
    results[name] = result
    print(f"{name:<20} {result['params_millions']:>8.2f}M      {result['memory_mb']:>8.1f}")

print("\nKey Observations:")
print(f"  - Q,V only: Minimal parameters, good for classification")
print(f"  - All Attention: 2x Q,V, standard choice")
print(f"  - Attention + FFN: 5-6x Q,V, best for generation")
print(f"  - FFN only: Rarely used, experimental")

In [ ]:
# Visualize parameter counts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

config_names = list(results.keys())
param_counts = [results[name]['params_millions'] for name in config_names]
memory_counts = [results[name]['memory_mb'] for name in config_names]
colors = ['skyblue', 'coral', 'lightgreen', 'gold']

# Parameters comparison
bars1 = ax1.barh(config_names, param_counts, color=colors, edgecolor='black', linewidth=2)
ax1.set_xlabel('Trainable Parameters (Millions)', fontsize=12)
ax1.set_title('LoRA Parameters by Configuration', fontsize=14, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for bar, count in zip(bars1, param_counts):
    width = bar.get_width()
    ax1.text(width, bar.get_y() + bar.get_height()/2,
            f'{count:.1f}M',
            ha='left', va='center', fontsize=10, fontweight='bold', 
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Memory comparison
bars2 = ax2.barh(config_names, memory_counts, color=colors, edgecolor='black', linewidth=2)
ax2.set_xlabel('Training Memory (MB)', fontsize=12)
ax2.set_title('Memory Requirements by Configuration', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for bar, mem in zip(bars2, memory_counts):
    width = bar.get_width()
    ax2.text(width, bar.get_y() + bar.get_height()/2,
            f'{mem:.0f} MB',
            ha='left', va='center', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('./lora_target_modules_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved!")

## Step 3: Ablation Study - Which Modules Matter?

### Experiment Design

We'll analyze the impact of different target modules on:
1. **Perplexity** (language modeling quality)
2. **Downstream task performance** (classification, generation)
3. **Training efficiency** (convergence speed)

### Expected Results (Based on Literature)

**Classification Tasks (GLUE, SuperGLUE):**
```
Configuration          Accuracy    Training Time
Q,V only              85.2%       1.0x (baseline)
Q,K,V,O (all attn)    85.7%       1.3x
Attention + FFN       85.9%       2.1x

Takeaway: Attention-only sufficient, FFN provides diminishing returns
```

**Generation Tasks (Text generation, summarization):**
```
Configuration          ROUGE-L     Perplexity
Q,V only              41.2        12.3
Q,K,V,O (all attn)    42.8        11.1
Attention + FFN       44.1        10.2

Takeaway: FFN important for generation quality
```

**Domain Adaptation:**
```
Configuration          Domain Transfer
Q,V only              +12.3%
Q,K,V,O (all attn)    +15.7%
Attention + FFN       +19.2%

Takeaway: Broader coverage helps domain transfer
```

In [ ]:
# Simulate ablation study results
# (In practice, you would train models with each configuration)

import pandas as pd

# Classification task (SST-2 sentiment)
classification_results = {
    "Configuration": ["Q,V only", "Q,K,V,O", "Attn+FFN", "Full FT"],
    "Accuracy (%)": [92.1, 92.5, 92.7, 93.0],
    "Training Time (min)": [15, 22, 38, 120],
    "Memory (GB)": [8, 10, 14, 60]
}

# Generation task (CNN/DM summarization)
generation_results = {
    "Configuration": ["Q,V only", "Q,K,V,O", "Attn+FFN", "Full FT"],
    "ROUGE-L": [41.2, 42.8, 44.1, 44.5],
    "Perplexity": [12.3, 11.1, 10.2, 9.8],
    "Memory (GB)": [8, 10, 14, 60]
}

df_class = pd.DataFrame(classification_results)
df_gen = pd.DataFrame(generation_results)

print("=== Classification Task (SST-2 Sentiment) ===")
print(df_class.to_string(index=False))

print("\n=== Generation Task (CNN/DM Summarization) ===")
print(df_gen.to_string(index=False))

print("\n=== Key Findings ===")
print("1. Classification: Q,V sufficient (92.1% vs 92.7% with Attn+FFN)")
print("2. Generation: FFN matters (41.2 → 44.1 ROUGE-L)")
print("3. Both: Diminishing returns vs full FT (within 1-2%)")
print("4. Memory: Q,V uses 50% less than Attn+FFN")

In [ ]:
# Visualize ablation results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

configs = df_class["Configuration"].tolist()
colors = ['skyblue', 'coral', 'lightgreen', 'gold']

# Classification accuracy
bars = ax1.bar(range(len(configs)), df_class["Accuracy (%)"], color=colors, edgecolor='black', linewidth=2)
ax1.set_xticks(range(len(configs)))
ax1.set_xticklabels(configs, rotation=45, ha='right')
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Classification: Accuracy by Configuration', fontsize=14, fontweight='bold')
ax1.set_ylim(90, 94)
ax1.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, df_class["Accuracy (%)"]):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Generation ROUGE-L
bars = ax2.bar(range(len(configs)), df_gen["ROUGE-L"], color=colors, edgecolor='black', linewidth=2)
ax2.set_xticks(range(len(configs)))
ax2.set_xticklabels(configs, rotation=45, ha='right')
ax2.set_ylabel('ROUGE-L Score', fontsize=12)
ax2.set_title('Generation: Quality by Configuration', fontsize=14, fontweight='bold')
ax2.set_ylim(40, 45)
ax2.grid(axis='y', alpha=0.3)
for bar, rouge in zip(bars, df_gen["ROUGE-L"]):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height,
            f'{rouge:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Memory comparison
x = np.arange(len(configs))
width = 0.35
bars1 = ax3.bar(x - width/2, df_class["Memory (GB)"], width, label='Classification', color='skyblue', edgecolor='black')
bars2 = ax3.bar(x + width/2, df_gen["Memory (GB)"], width, label='Generation', color='coral', edgecolor='black')
ax3.set_xticks(x)
ax3.set_xticklabels(configs, rotation=45, ha='right')
ax3.set_ylabel('Memory (GB)', fontsize=12)
ax3.set_title('Memory Requirements', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# Efficiency plot: Quality vs Memory
class_quality = df_class["Accuracy (%)"].values
gen_quality = df_gen["ROUGE-L"].values
memory = df_class["Memory (GB)"].values

# Normalize scores
class_norm = (class_quality - 90) / (94 - 90) * 100
gen_norm = (gen_quality - 40) / (45 - 40) * 100

ax4.scatter(memory[:3], class_norm[:3], s=300, label='Classification', marker='o', 
           color=colors[:3], edgecolors='black', linewidth=2, alpha=0.7)
ax4.scatter(memory[:3], gen_norm[:3], s=300, label='Generation', marker='s',
           color=colors[:3], edgecolors='black', linewidth=2, alpha=0.7)

for i, config in enumerate(configs[:3]):
    ax4.annotate(config, (memory[i], class_norm[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

ax4.set_xlabel('Memory (GB)', fontsize=12)
ax4.set_ylabel('Normalized Quality Score', fontsize=12)
ax4.set_title('Efficiency: Quality vs Memory', fontsize=14, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(5, 20)

plt.tight_layout()
plt.savefig('./lora_ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAblation study visualization saved!")

## Step 4: Architecture-Specific Considerations

### Different Models, Different Module Names

**BERT / RoBERTa / ALBERT:**
```python
target_modules = [
    "query",     # Query projection
    "key",       # Key projection (optional)
    "value",     # Value projection
    "dense"      # Output projection (optional)
]
```

**GPT-2 / GPT-J:**
```python
target_modules = [
    "c_attn",    # Combined Q,K,V projection (split after)
    "c_proj",    # Output projection
    "c_fc",      # FFN intermediate (optional)
    "c_proj"     # FFN output (optional)
]
```

**Llama / Llama 2 / Mistral:**
```python
target_modules = [
    "q_proj",      # Query
    "k_proj",      # Key
    "v_proj",      # Value
    "o_proj",      # Output
    "gate_proj",   # FFN gate (SwiGLU)
    "up_proj",     # FFN up
    "down_proj"    # FFN down
]
```

**T5 / FLAN-T5:**
```python
# Encoder and decoder have different modules
encoder_modules = ["q", "v"]  # Encoder attention
decoder_modules = ["q", "v"]  # Decoder self-attention
cross_modules = ["q", "v"]    # Cross-attention
```

In [ ]:
# Helper function to find target modules automatically
def find_linear_module_names(model) -> List[str]:
    """
    Automatically find all linear layer names in a model.
    Useful for discovering target modules in unfamiliar architectures.
    """
    linear_module_names = set()
    
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            # Extract the module name (last component)
            module_name = name.split('.')[-1]
            linear_module_names.add(module_name)
    
    return sorted(list(linear_module_names))

# Test with Llama model
print("=== Discovering Linear Modules in TinyLlama ===\n")
linear_modules = find_linear_module_names(model)

print("Found linear modules:")
for i, module_name in enumerate(linear_modules, 1):
    print(f"  {i}. {module_name}")

print("\nRecommended configurations:")
print("  Minimal: ['q_proj', 'v_proj']")
print("  Standard: ['q_proj', 'k_proj', 'v_proj', 'o_proj']")
print("  Full: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']")

# Test with different model architectures
model_architectures = {
    "TinyLlama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    # Add more as needed
}

print("\n=== Module Names Across Architectures ===")
for arch_name, model_id in model_architectures.items():
    print(f"\n{arch_name}:")
    # In practice, you would load and inspect each model
    if arch_name == "TinyLlama":
        print("  Attention: q_proj, k_proj, v_proj, o_proj")
        print("  FFN: gate_proj, up_proj, down_proj")

## Step 5: Practical Recommendations

### Decision Tree for Target Module Selection

```
START: Which task are you fine-tuning for?

├── Classification / Q&A / Named Entity Recognition
│   └── Use: ["q_proj", "v_proj"]
│       - Fast training
│       - Low memory
│       - 95-98% of full attention quality
│
├── Text Generation / Summarization / Translation  
│   ├── Memory < 16 GB?
│   │   └── Use: ["q_proj", "k_proj", "v_proj", "o_proj"]
│   │       - Balanced quality/memory
│   └── Memory >= 16 GB?
│       └── Use: All attention + FFN
│           - Best generation quality
│
├── Domain Adaptation (Medical, Legal, Code)
│   └── Use: All attention + FFN
│       - Captures domain-specific patterns
│       - Better transfer learning
│
└── Instruction Tuning (Alpaca, Vicuna-style)
    └── Use: All attention (+ FFN if memory allows)
        - Improves instruction following
        - Better generalization
```

### Configuration Recipes

**Recipe 1: Memory-Constrained Classification**
```python
lora_config = LoraConfig(
    r=8,                              # Lower rank
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Minimal
    lora_dropout=0.1,
    task_type=TaskType.SEQ_CLS
)
# Expected: ~2M params, 8-16 MB memory
```

**Recipe 2: Balanced General-Purpose**
```python
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # All attention
    lora_dropout=0.1,
    task_type=TaskType.CAUSAL_LM
)
# Expected: ~4M params, 16-32 MB memory
```

**Recipe 3: Quality-Focused Generation**
```python
lora_config = LoraConfig(
    r=32,                             # Higher rank
    lora_alpha=64,
    target_modules=[                  # Everything
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,                # Lower dropout
    task_type=TaskType.CAUSAL_LM
)
# Expected: ~20M params, 80-160 MB memory
```

In [ ]:
# Create configurations and compare
def create_config_comparison():
    """Compare different LoRA configurations."""
    
    configs = {
        "Memory-Constrained": {
            "r": 8,
            "alpha": 16,
            "target_modules": ["q_proj", "v_proj"],
            "use_case": "Classification, Q&A",
            "quality": "Good (95%)",
            "memory": "Minimal"
        },
        "Balanced": {
            "r": 16,
            "alpha": 32,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
            "use_case": "General purpose",
            "quality": "Very Good (98%)",
            "memory": "Moderate"
        },
        "Quality-Focused": {
            "r": 32,
            "alpha": 64,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", 
                             "gate_proj", "up_proj", "down_proj"],
            "use_case": "Generation, domain adapt",
            "quality": "Excellent (99%)",
            "memory": "High"
        }
    }
    
    print("=== LoRA Configuration Comparison ===\n")
    for name, config in configs.items():
        print(f"{name}:")
        print(f"  Rank: {config['r']}")
        print(f"  Target modules: {len(config['target_modules'])} modules")
        print(f"  Best for: {config['use_case']}")
        print(f"  Quality: {config['quality']}")
        print(f"  Memory: {config['memory']}")
        print()
    
    return configs

comparison = create_config_comparison()

print("=== Quick Selection Guide ===")
print("Budget GPU (< 12 GB):  Memory-Constrained config")
print("Standard GPU (24 GB):  Balanced config")
print("High-end GPU (40+ GB): Quality-Focused config")

## Step 6: Advanced: Module-Specific Ranks

### Different Ranks for Different Modules

Recent research shows that different modules may benefit from different ranks:

```python
# Advanced: Module-specific configuration
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,  # Default rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "o_proj"],
    
    # Override rank for specific modules
    rank_pattern={
        "q_proj": 32,    # Higher rank for queries (more important)
        "v_proj": 32,    # Higher rank for values
        "o_proj": 8      # Lower rank for output (less critical)
    },
    
    task_type=TaskType.CAUSAL_LM
)
```

**Benefits:**
- Allocate capacity where it matters most
- Optimize parameter budget
- Improve quality without increasing total parameters

**Typical Patterns:**
- Q, V projections: Higher rank (16-32)
- K, O projections: Lower rank (8-16)
- FFN: Medium rank (16)

In [ ]:
# Simulate module-specific rank configuration
def compare_rank_strategies():
    """Compare uniform vs module-specific ranks."""
    
    hidden_size = 4096
    num_layers = 32
    
    # Strategy 1: Uniform rank
    uniform_rank = 16
    uniform_modules = ["q_proj", "v_proj", "k_proj", "o_proj"]
    uniform_params = len(uniform_modules) * 2 * hidden_size * uniform_rank * num_layers
    
    # Strategy 2: Module-specific ranks
    specific_ranks = {
        "q_proj": 32,   # Most important
        "v_proj": 32,   # Most important
        "k_proj": 8,    # Less critical
        "o_proj": 8     # Less critical
    }
    specific_params = sum(
        2 * hidden_size * rank * num_layers 
        for module, rank in specific_ranks.items()
    )
    
    print("=== Rank Strategy Comparison ===\n")
    print("Uniform Rank Strategy:")
    print(f"  All modules: r={uniform_rank}")
    print(f"  Total params: {uniform_params/1e6:.2f}M")
    print()
    print("Module-Specific Rank Strategy:")
    for module, rank in specific_ranks.items():
        print(f"  {module}: r={rank}")
    print(f"  Total params: {specific_params/1e6:.2f}M")
    print()
    print(f"Parameter savings: {uniform_params/specific_params:.2f}x")
    print("Quality: Similar or better (allocate budget efficiently)")

compare_rank_strategies()

## Summary: LoRA Target Module Selection

### What We Learned

1. **Attention vs FFN:**
   - Attention: 33% of parameters, critical for most tasks
   - FFN: 67% of parameters, important for generation

2. **Task-Specific Recommendations:**
   - Classification: Q,V sufficient
   - Generation: All attention + FFN
   - Domain adaptation: Broader coverage helps

3. **Memory Trade-offs:**
   - Q,V only: 2-4M params
   - All attention: 4-8M params
   - Attention + FFN: 20-30M params

### Quick Reference Table

| Task Type | Target Modules | Rank | Quality | Memory |
|-----------|---------------|------|---------|--------|
| Classification | q_proj, v_proj | 8-16 | 95% | Low |
| Q&A | q_proj, v_proj | 8-16 | 96% | Low |
| Generation | All attention | 16-32 | 98% | Medium |
| Generation (high quality) | Attn + FFN | 32-64 | 99% | High |
| Domain Adaptation | Attn + FFN | 16-32 | 97% | Medium-High |
| Instruction Tuning | All attention | 16-32 | 97% | Medium |

### Best Practices

1. **Start with all attention (Q,K,V,O)**
   - Good default for most tasks
   - Balanced quality/memory

2. **Use Q,V only if:**
   - Memory constrained (< 12 GB)
   - Simple classification task
   - Quality > 95% is sufficient

3. **Add FFN if:**
   - Generation quality matters
   - Domain adaptation needed
   - Memory not constrained

4. **Experiment with module-specific ranks:**
   - Allocate higher ranks to Q,V
   - Lower ranks to K,O
   - Can improve efficiency

### Architecture-Specific Module Names

| Architecture | Q | K | V | O | FFN |
|-------------|---|---|---|---|-----|
| Llama/Mistral | q_proj | k_proj | v_proj | o_proj | gate/up/down_proj |
| GPT-2 | c_attn | c_attn | c_attn | c_proj | c_fc, c_proj |
| BERT | query | key | value | dense | intermediate, output |
| T5 | q | k | v | o | wi, wo |

### Next Steps

1. **Notebook 24**: Custom loss functions with LoRA
   - All Stage 1 losses work with LoRA
   - Weighted CE, focal loss
   - Multi-task learning

### References

- Hu et al. (2021): "LoRA: Low-Rank Adaptation of Large Language Models"
- Various community ablation studies on GitHub/HuggingFace
- PEFT documentation: https://huggingface.co/docs/peft